In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/jauharulumam/permenkes-10-2024/permenkes-no-10-tahun-2024.pdf


In [2]:
pip install PyMuPDF

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.8/25.8 MB 61.4 MB/s eta 0:00:00:00:0100:01
Note: you may need to restart the kernel to use updated packages.


In [19]:
import fitz
import re


def extract_peraturan_pdf(pdf_path):
    doc = fitz.open(pdf_path)

    all_pages = []

    for page in doc:
        text = page.get_text("text")

        # Normalisasi
        text = text.replace("\xa0", " ")
        text = text.replace("\u00ad", "")

        # Hapus nomor halaman
        text = re.sub(r'(?m)^\s*-\d+-\s*$', '', text)

        # Pecah berdasarkan baris
        lines = [line.strip() for line in text.splitlines()]
        lines = [line for line in lines if line]

        result = []
        paragraph = ""

        for line in lines:

            # Heading Pasal
            if re.match(r'^Pasal\s+\d+', line, re.IGNORECASE):
                if paragraph:
                    result.append(paragraph)
                    paragraph = ""

                result.append(line)
                continue

            # Ayat: (1), (2), dst.
            if re.match(r'^\(\d+\)', line):
                if paragraph:
                    result.append(paragraph)
                    paragraph = ""

                paragraph = line
                continue

            # Huruf: a., b., c., dst.
            if re.match(r'^[a-z]\.', line):
                if paragraph:
                    result.append(paragraph)
                    paragraph = ""

                paragraph = line
                continue

            # Kalau bukan struktur baru,
            # berarti kemungkinan lanjutan baris sebelumnya
            if paragraph:
                paragraph += " " + line
            else:
                paragraph = line

        if paragraph:
            result.append(paragraph)

        all_pages.extend(result)

    return "\n\n".join(all_pages)

In [20]:
text = extract_peraturan_pdf("/kaggle/input/datasets/jauharulumam/permenkes-10-2024/permenkes-no-10-tahun-2024.pdf")

print(text)

PERATURAN MENTERI KESEHATAN REPUBLIK INDONESIA NOMOR 10 TAHUN 2024 TENTANG JARINGAN DOKUMENTASI DAN INFORMASI HUKUM DI LINGKUNGAN KEMENTERIAN KESEHATAN DENGAN RAHMAT TUHAN YANG MAHA ESA MENTERI KESEHATAN REPUBLIK INDONESIA, Menimbang : a. bahwa untuk meningkatkan pelayanan kepada masyarakat terhadap kebutuhan dokumentasi dan informasi hukum bidang kesehatan secara lengkap, akurat, mudah, dan cepat perlu pengelolaan jaringan dokumentasi dan informasi hukum yang tertata dan terselenggara dengan baik;

b. bahwa berdasarkan pertimbangan sebagaimana dimaksud dalam huruf a dan untuk melaksanakan ketentuan Pasal 5 ayat (1) Peraturan Presiden Nomor 33 Tahun 2012 tentang Jaringan Dokumentasi dan Informasi Hukum Nasional, perlu menetapkan Peraturan Menteri Kesehatan tentang Jaringan Dokumentasi dan Informasi Hukum di Lingkungan Kementerian Kesehatan; Mengingat : 1.

Pasal 17 ayat (3) Undang-Undang Dasar Negara Republik

Indonesia Tahun 1945; 2. Undang-Undang Nomor 39 Tahun 2008 tentang Kementeri

In [21]:
import re


def split_per_pasals(text):
    # Pasal harus berada di awal baris
    # dan diikuti nomor
    pattern = r'(?m)(?=^Pasal\s+\d+\s*$)'

    pasals = re.split(pattern, text)

    # Buang bagian sebelum Pasal 1
    pasals = [
        p.strip()
        for p in pasals
        if re.match(r'^Pasal\s+\d+', p.strip())
    ]

    return pasals

In [22]:
pasals = split_per_pasals(text)

for pasal in pasals:
    print("=" * 50)
    print(pasal)

Pasal 1

Dalam Peraturan Menteri ini yang dimaksud dengan: 1. Jaringan Dokumentasi dan Informasi Hukum Nasional yang selanjutnya disingkat JDIHN adalah wadah pendayagunaan bersama atas dokumen hukum secara tertib, terpadu, dan berkesinambungan, serta merupakan sarana pemberian pelayanan informasi hukum secara lengkap, akurat, mudah, dan cepat. 2. Jaringan Dokumentasi dan Informasi Hukum di lingkungan Kementerian Kesehatan yang selanjutnya disebut JDIH Kemenkes adalah suatu sistem pengelolaan dan pendayagunaan bersama dokumen hukum dan informasi hukum di bidang kesehatan secara tertib, terpadu dan berkesinambungan serta merupakan sarana pemberian pelayanan informasi hukum secara lengkap, akurat, mudah dan cepat. 3. Dokumen Hukum adalah produk hukum yang berupa peraturan perundang-undangan atau produk hukum selain peraturan perundang-undangan yang meliputi namun tidak terbatas pada putusan pengadilan, yurisprudensi, monografi hukum, artikel majalah hukum, buku hukum, penelitian hukum, pe

# **Generate QA**

In [31]:
import json
import os
from openai import OpenAI

from kaggle_secrets import UserSecretsClient
secret_label = "OPENAI_API_KEY"
secret_value = UserSecretsClient().get_secret(secret_label)
# Pastikan Anda sudah set environment variable OPENAI_API_KEY
# atau masukkan api_key="sk-..." langsung ke dalam OpenAI()
client = OpenAI(api_key=secret_value, timeout=900.0)

def generate_qa_pairs(teks_hukum, konteks_referensi=""):
    system_prompt = """
Anda adalah seorang ahli dalam bidang hukum dan peraturan perundang-undangan Indonesia sekaligus penyusun dataset untuk fine-tuning Large Language Model (LLM).
Tugas Anda adalah membuat pasangan Question-Answer (QA) berdasarkan teks peraturan yang diberikan.

TUJUAN:
Menghasilkan dataset QA berkualitas tinggi yang membantu model bahasa kecil memahami dan menjawab pertanyaan mengenai isi peraturan perundang-undangan Indonesia secara akurat.

ATURAN UTAMA:
1. Gunakan HANYA informasi yang terdapat dalam teks peraturan dan context referensi yang diberikan.
2. Jangan menambahkan, menyimpulkan, atau mengarang informasi yang tidak secara jelas didukung oleh teks.
3. Pertahankan makna hukum dari teks sumber.
4. Buat pertanyaan yang benar-benar memiliki informasi berbeda atau menguji cara pemahaman yang berbeda. Semakin banyak pertanyaan lebih baik dengan catatan 
5. Tentukan jumlah QA berdasarkan kepadatan dan kompleksitas informasi dalam Pasal.
6. Hasilkan pertanyaan sebanyak mungkin (semakin banyak semakin baik) untuk mengekstrak seluruh informasi dari teks. Syarat mutlak: setiap pertanyaan harus menguji substansi yang berbeda dan bukan sekadar memparafrase pertanyaan yang sudah ada (mengacu pada aturan nomor 4).
7. Untuk Pasal yang mengandung banyak informasi penting, dapat dibuat hingga maksimal 25 QA.
8. Jangan membuat QA hanya untuk memperbanyak jumlah dataset.
9. Pertanyaan harus terdengar seperti pertanyaan nyata.
10. Variasikan bentuk pertanyaan (langsung, rincian, ya/tidak, dll).
11. Jangan membuat pertanyaan yang jawabannya tidak dapat ditemukan dari teks.
12. Jika ada rujukan ("sebagaimana dimaksud...") dan tersedia di REFERENCE CONTEXT, gunakan informasi tersebut.
13. Jika rujukan tidak tersedia di context, jangan mengarang isinya.
14. Jangan memecah setiap item daftar menjadi satu QA secara otomatis, jadikan satu pertanyaan menyeluruh kecuali ada kebutuhan spesifik.
15. Jawaban harus langsung menjawab pertanyaan, lengkap, akurat, dan wajib menyertakan referensi sumber pasal secara natural dan bervariasi. Variasikan letak dan gaya penyebutan sumber referensi.
16. Jika pertanyaan meminta daftar, jawaban harus mencantumkan seluruh item.
17. Jika pertanyaan ya/tidak, sebutkan "Ya" atau "Tidak" lalu beri penjelasan.
18. Wajib sertakan minimal satu pertanyaan yang secara spesifik menanyakan bunyi atau isi dari suatu Pasal (misalnya: "Apa isi dari Pasal...", "Sebutkan bunyi Pasal..."). Jawaban untuk pertanyaan jenis ini HARUS disalin sama persis dari teks peraturan asli.
19. Jangan menggunakan pengetahuan umum tentang hukum Indonesia untuk melengkapi jawaban.
20. Setiap QA harus dapat ditelusuri kembali ke Pasal/Ayat sumber.
21. Dilarang menyebutkan referensi nomor Pasal, Ayat, atau Huruf di dalam teks pertanyaan (contoh SALAH: "Menurut Pasal 2, apa tujuan..."). Asumsikan penanya menanyakan topik tanpa mengetahui letak aturan tersebut di dokumen. PENGECUALIAN mutlak HANYA berlaku untuk pertanyaan dengan tipe exact_text (contoh: "Sebutkan bunyi Pasal 2").

FORMAT OUTPUT:
Output HARUS berupa JSON valid dengan satu key utama "dataset" yang berisi array of objects.
{
  "dataset": [
    {
      "pasal": "...",
      "qa_pairs": [
        {
          "question": "...",
          "answer": "...",
          "question_type": "...",
          "sumber_ayat": "..." // Isi dengan ayat spesifik (misal: "Ayat 1"), atau null jika merujuk pada keseluruhan Pasal
        }
      ]
    }
  ]
}
Jangan memasukkan markdown di luar JSON.
"""

    user_prompt = f"""
Buat pasangan QA dari teks peraturan berikut:

<TEKS_PERATURAN>
{teks_hukum}
</TEKS_PERATURAN>

<REFERENCE_CONTEXT>
{konteks_referensi}
</REFERENCE_CONTEXT>
"""

    try:
        response = client.chat.completions.create(
            model="gpt-6-astra", # Hemat biaya dan sangat efisien untuk ekstraksi
            service_tier="flex",
            messages=[
                {"role": "system", "content": system_prompt.strip()},
                {"role": "user", "content": user_prompt.strip()}
            ],
            response_format={"type": "json_object"},
            max_completion_tokens=3000
        )
        
        # Ekstrak string JSON dari response
        raw_json = response.choices[0].message.content
        
        # Parsing string menjadi dictionary Python
        parsed_data = json.loads(raw_json)
        return parsed_data
        
    except Exception as e:
        print(f"Terjadi kesalahan saat memanggil API: {e}")
        return None



In [28]:
# --- CONTOH PENGGUNAAN ---
if __name__ == "__main__":
    # Teks sampel dari dokumen PDF Anda (misal Permenkes No 10 Tahun 2024)
    sampel_pasal = f"""
    {pasals[0]}
    """
    
    hasil = generate_qa_pairs(sampel_pasal)
    
    if hasil:
        # Cetak hasil ekstraksi
        print(json.dumps(hasil, indent=2, ensure_ascii=False))


{
  "dataset": [
    {
      "pasal": "1",
      "ayat": null,
      "qa_pairs": [
        {
          "question": "Apa bunyi Pasal 1 yang memuat pengertian istilah dalam Peraturan Menteri ini?",
          "answer": "Pasal 1\n\nDalam Peraturan Menteri ini yang dimaksud dengan: 1. Jaringan Dokumentasi dan Informasi Hukum Nasional yang selanjutnya disingkat JDIHN adalah wadah pendayagunaan bersama atas dokumen hukum secara tertib, terpadu, dan berkesinambungan, serta merupakan sarana pemberian pelayanan informasi hukum secara lengkap, akurat, mudah, dan cepat. 2. Jaringan Dokumentasi dan Informasi Hukum di lingkungan Kementerian Kesehatan yang selanjutnya disebut JDIH Kemenkes adalah suatu sistem pengelolaan dan pendayagunaan bersama dokumen hukum dan informasi hukum di bidang kesehatan secara tertib, terpadu dan berkesinambungan serta merupakan sarana pemberian pelayanan informasi hukum secara lengkap, akurat, mudah dan cepat. 3. Dokumen Hukum adalah produk hukum yang berupa peraturan p

In [32]:
import json
import time


# 1. Buat list kosong untuk menampung seluruh dataset dari semua pasal
semua_dataset = []

print(f"Ditemukan {len(pasals)} pasal untuk diproses.\n")

for i, pasal in enumerate(pasals, start=1):
    print("=" * 50)
    print(f"Memproses Pasal ke-{i}...")
    
    # 2. Panggil API untuk pasal saat ini
    hasil = generate_qa_pairs(teks_hukum=pasal)
    
    # 3. Validasi hasil (jika API sukses dan mengembalikan data)
    if hasil and "dataset" in hasil:
        # Gunakan extend() untuk menggabungkan isi array, bukan append()
        semua_dataset.extend(hasil["dataset"])
        
        # Hitung jumlah QA yang didapat untuk log
        jumlah_qa = sum(len(item.get("qa_pairs", [])) for item in hasil["dataset"])
        print(f"[SUKSES] Ekstrak {jumlah_qa} QA pairs.")
    else:
        print(f"[GAGAL] Gagal mengekstrak QA dari Pasal ke-{i}. Lanjut ke pasal berikutnya.")
    
    # 4. Beri jeda waktu (PENTING untuk menghindari RateLimitError dari OpenAI)
    # Sesuaikan jeda berdasarkan tier akun OpenAI Anda (misal 2 detik)
    time.sleep(2)

print("=" * 50)
print(f"Semua pasal selesai diproses. Total keseluruhan struktur pasal: {len(semua_dataset)}")

# 5. Simpan hasil akhir ke dalam file JSON
nama_file = "dataset_qa_permenkes_astra.json"
with open(nama_file, "w", encoding="utf-8") as f:
    json.dump(semua_dataset, f, indent=2, ensure_ascii=False)

print(f"Dataset mentah berhasil disimpan ke {nama_file}")

Ditemukan 11 pasal untuk diproses.

Memproses Pasal ke-1...
[SUKSES] Ekstrak 10 QA pairs.
Memproses Pasal ke-2...
[SUKSES] Ekstrak 4 QA pairs.
Memproses Pasal ke-3...
[SUKSES] Ekstrak 5 QA pairs.
Memproses Pasal ke-4...
[SUKSES] Ekstrak 8 QA pairs.
Memproses Pasal ke-5...
[SUKSES] Ekstrak 4 QA pairs.
Memproses Pasal ke-6...
[SUKSES] Ekstrak 8 QA pairs.
Memproses Pasal ke-7...
[SUKSES] Ekstrak 5 QA pairs.
Memproses Pasal ke-8...
[SUKSES] Ekstrak 13 QA pairs.
Memproses Pasal ke-9...
[SUKSES] Ekstrak 4 QA pairs.
Memproses Pasal ke-10...
[SUKSES] Ekstrak 3 QA pairs.
Memproses Pasal ke-11...
[SUKSES] Ekstrak 6 QA pairs.
Semua pasal selesai diproses. Total keseluruhan struktur pasal: 11
Dataset mentah berhasil disimpan ke dataset_qa_permenkes_astra.json
